In [9]:
import re
import numpy as np
import pandas as pd
import itertools
from tqdm.auto import tqdm


import matplotlib.pyplot as plt


from bert_score import score
import pymorphy3
from rapidfuzz.distance import Levenshtein
from sacrebleu.metrics import BLEU, CHRF

In [10]:
files = {
    "synonyms": "c2_augmented_synonyms.csv",
    "antonyms": "c2_augmented_antonyms.csv",
}

# Вспомогательные функции подсчета метрик

In [11]:
morph = pymorphy3.MorphAnalyzer()

def tokenize_words(text):
    text = str(text).lower()
    return re.findall(r"[а-яёa-z]+", text)


def lemmatize_text(text):
    words = tokenize_words(text)
    lemmas = [morph.parse(word)[0].normal_form for word in words]
    return lemmas

In [12]:
def bertscore_pair(original_text, augmented_text, model_type="DeepPavlov/rubert-base-cased"):
    """
    Считает BERTScore между двумя текстами:
    original_text — исходный текст
    augmented_text — сгенерированный / аугментированный текст
    """

    P, R, F1 = score(
        [augmented_text],      # candidate
        [original_text],       # reference
        model_type=model_type,
        num_layers=12,
        lang="ru",
        verbose=False
    )

    return {
        "precision": P.item(),
        "recall": R.item(),
        "f1": F1.item()
    }

In [13]:
bleu_metric = BLEU(effective_order=True)
chrf_metric = CHRF()


def jaccard_similarity_lemmas(original, augmented):
    orig_lemmas = set(lemmatize_text(original))
    aug_lemmas = set(lemmatize_text(augmented))

    if not orig_lemmas and not aug_lemmas:
        return 1.0

    if not orig_lemmas or not aug_lemmas:
        return 0.0

    return len(orig_lemmas & aug_lemmas) / len(orig_lemmas | aug_lemmas)


def common_words_ratio(original, augmented):
    """
    Доля лемм исходного текста, которые сохранились в аугментированном.
    """
    orig_lemmas = set(lemmatize_text(original))
    aug_lemmas = set(lemmatize_text(augmented))

    if not orig_lemmas:
        return 0.0

    return len(orig_lemmas & aug_lemmas) / len(orig_lemmas)


def normalized_levenshtein_distance(original, augmented):
    """
    0 — тексты одинаковые.
    1 — тексты максимально различаются.
    """
    original = str(original)
    augmented = str(augmented)

    max_len = max(len(original), len(augmented))

    if max_len == 0:
        return 0.0

    distance = Levenshtein.distance(original, augmented)
    return distance / max_len


def normalized_levenshtein_similarity(original, augmented):
    """
    1 — тексты одинаковые.
    0 — тексты максимально различаются.
    """
    return 1 - normalized_levenshtein_distance(original, augmented)

In [14]:
def lcs_length(x, y):
    """
    Longest Common Subsequence для двух списков токенов.
    """
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_f1(original, augmented):
    orig_tokens = tokenize_words(original)
    aug_tokens = tokenize_words(augmented)

    if not orig_tokens or not aug_tokens:
        return 0.0

    lcs = lcs_length(orig_tokens, aug_tokens)

    precision = lcs / len(aug_tokens)
    recall = lcs / len(orig_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

In [15]:
def bleu_score(original, augmented):
    """
    BLEU: насколько n-граммы аугментированного текста совпадают с исходным.
    Значение приводим к диапазону 0–1.
    """
    score = bleu_metric.sentence_score(
        hypothesis=str(augmented),
        references=[str(original)]
    ).score

    return score / 100


def chrf_score(original, augmented):
    """
    chrF: сходство на уровне символьных n-грамм.
    Значение приводим к диапазону 0–1.
    """
    score = chrf_metric.sentence_score(
        hypothesis=str(augmented),
        references=[str(original)]
    ).score

    return score / 100

# Сравнение исходных и аугментированных текстов внутри одной температуры

In [16]:
def calculate_pair_metrics(original, augmented):
    return {
        "bert_score": bertscore_pair(original, augmented),
        "jaccard_lemmas": jaccard_similarity_lemmas(original, augmented),
        "common_words_ratio": common_words_ratio(original, augmented),
        "levenshtein_distance": normalized_levenshtein_distance(original, augmented),
        "levenshtein_similarity": normalized_levenshtein_similarity(original, augmented),
        "rouge_l": rouge_l_f1(original, augmented),
        "bleu": bleu_score(original, augmented),
        "chrf": chrf_score(original, augmented),
    }

In [17]:
all_rows = []

for augmentation_type, path in files.items():
    df = pd.read_csv(path)

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Обработка строк"):
        original = row['text']
        augmented = row['augmented-text']

        metrics = calculate_pair_metrics(original, augmented)

        metrics["augmentation_type"] = augmentation_type
        metrics["original"] = original
        metrics["augmented"] = augmented

        all_rows.append(metrics)

pair_metrics_df = pd.DataFrame(all_rows)

Обработка строк:   0%|          | 0/119 [00:00<?, ?it/s]

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: 

Обработка строк:   0%|          | 0/71 [00:00<?, ?it/s]

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: 

In [18]:
pair_metrics_df

,bert_score,jaccard_lemmas,common_words_ratio,levenshtein_distance,levenshtein_similarity,rouge_l,bleu,chrf,augmentation_type,original,augmented
0,"{'precision': 0.9480492472648621, 'recall': 0....",0.846154,0.916667,0.089888,0.910112,0.916667,0.747737,0.916163,synonyms,"3:38–3:54 Собственно, сама по себе радиация не...","3: 38 – 3: 54 Собственно, сама по себе радиаци..."
1,"{'precision': 0.8833481073379517, 'recall': 0....",0.733333,0.846154,0.165899,0.834101,0.814815,0.584491,0.776470,synonyms,Недавно компания Uber объявила об инвестиции о...,Недавно фирма Uber объявила об инвестиции одно...
2,"{'precision': 0.9230732917785645, 'recall': 0....",0.818182,0.900000,0.078313,0.921687,0.900000,0.824636,0.895030,synonyms,"Множество повестей: «Двойник», «Дядюшкин сон»,...","Множество повестей: «Двойник», «Дядюшкин сон»,..."
3,"{'precision': 0.8132451772689819, 'recall': 0....",0.500000,0.666667,0.193878,0.806122,0.680000,0.307616,0.684365,synonyms,Встречи одноклассников и одногруппников превра...,Встречи одноклассников и одногруппников преобр...
4,"{'precision': 0.7609840631484985, 'recall': 0....",0.350000,0.518519,0.295833,0.704167,0.517241,0.288958,0.594653,synonyms,Самопрезентация — как главная черта времени и ...,Самопрезентация — как основная грань времени и...
...,...,...,...,...,...,...,...,...,...,...,...
185,"{'precision': 0.9429249167442322, 'recall': 0....",0.935484,0.966667,0.037190,0.962810,0.966667,0.927898,0.959564,antonyms,"При этом определение того, что является «разум...","При этом определение того, что является «бессм..."
186,"{'precision': 1.0, 'recall': 1.0, 'f1': 1.0}",1.000000,1.000000,0.006803,0.993197,1.000000,0.871393,1.000000,antonyms,"От одного банана — в два раза больше (0,1), а ...","От одного банана — в два раза больше (0, 1), а..."
187,"{'precision': 0.9426813125610352, 'recall': 0....",0.920000,0.958333,0.035176,0.964824,0.961538,0.943623,0.954629,antonyms,00:00:24 А сегодня испытания. 13 участников. Ж...,00: 00: 24 А сегодня испытания. 13 участников....
188,"{'precision': 0.9767154455184937, 'recall': 0....",0.928571,0.962963,0.046948,0.953052,0.965517,0.918468,0.942921,antonyms,«Прома» способна предложить дилерам вполне бож...,«Прома» способна предложить дилерам вполне бож...


In [20]:
pair_metrics_df['bert_score_precision'] = [pair_metrics_df['bert_score'][i]['precision'] for i in range(len(pair_metrics_df))]
pair_metrics_df['bert_score_recall'] = [pair_metrics_df['bert_score'][i]['recall'] for i in range(len(pair_metrics_df))]
pair_metrics_df['bert_score_f1'] = [pair_metrics_df['bert_score'][i]['f1'] for i in range(len(pair_metrics_df))]

del pair_metrics_df['bert_score']

pair_metrics_df

,jaccard_lemmas,common_words_ratio,levenshtein_distance,levenshtein_similarity,rouge_l,bleu,chrf,augmentation_type,original,augmented,bert_score_precision,bert_score_recall,bert_score_f1
0,0.846154,0.916667,0.089888,0.910112,0.916667,0.747737,0.916163,synonyms,"3:38–3:54 Собственно, сама по себе радиация не...","3: 38 – 3: 54 Собственно, сама по себе радиаци...",0.948049,0.957350,0.952677
1,0.733333,0.846154,0.165899,0.834101,0.814815,0.584491,0.776470,synonyms,Недавно компания Uber объявила об инвестиции о...,Недавно фирма Uber объявила об инвестиции одно...,0.883348,0.883348,0.883348
2,0.818182,0.900000,0.078313,0.921687,0.900000,0.824636,0.895030,synonyms,"Множество повестей: «Двойник», «Дядюшкин сон»,...","Множество повестей: «Двойник», «Дядюшкин сон»,...",0.923073,0.932512,0.927769
3,0.500000,0.666667,0.193878,0.806122,0.680000,0.307616,0.684365,synonyms,Встречи одноклассников и одногруппников превра...,Встречи одноклассников и одногруппников преобр...,0.813245,0.824280,0.818725
4,0.350000,0.518519,0.295833,0.704167,0.517241,0.288958,0.594653,synonyms,Самопрезентация — как главная черта времени и ...,Самопрезентация — как основная грань времени и...,0.760984,0.821450,0.790062
...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,0.935484,0.966667,0.037190,0.962810,0.966667,0.927898,0.959564,antonyms,"При этом определение того, что является «разум...","При этом определение того, что является «бессм...",0.942925,0.942925,0.942925
186,1.000000,1.000000,0.006803,0.993197,1.000000,0.871393,1.000000,antonyms,"От одного банана — в два раза больше (0,1), а ...","От одного банана — в два раза больше (0, 1), а...",1.000000,1.000000,1.000000
187,0.920000,0.958333,0.035176,0.964824,0.961538,0.943623,0.954629,antonyms,00:00:24 А сегодня испытания. 13 участников. Ж...,00: 00: 24 А сегодня испытания. 13 участников....,0.942681,0.971205,0.956731
188,0.928571,0.962963,0.046948,0.953052,0.965517,0.918468,0.942921,antonyms,«Прома» способна предложить дилерам вполне бож...,«Прома» способна предложить дилерам вполне бож...,0.976715,0.978997,0.977855


In [21]:
metric_columns = [
    "bert_score_precision",
    "bert_score_recall",
    "bert_score_f1",
    "jaccard_lemmas",
    "common_words_ratio",
    "levenshtein_distance",
    "levenshtein_similarity",
    "rouge_l",
    "bleu",
    "chrf",
]

results_df = (
    pair_metrics_df
    .groupby("augmentation_type")[metric_columns]
    .agg(["mean", "std", "min", "max"])
)

In [22]:
results_df_flat = results_df.copy()

results_df_flat.columns = [
    f"{metric}_{stat}"
    for metric, stat in results_df_flat.columns
]

results_df_flat = results_df_flat.reset_index()

In [23]:
results_df['bert_score_precision']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.960420,0.035449,0.832547,1.000000
synonyms,0.846712,0.067434,0.624613,0.984292


In [24]:
results_df['bert_score_recall']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.969240,0.028873,0.867680,1.000000
synonyms,0.874106,0.057577,0.699298,0.984292


In [25]:
results_df['bert_score_f1']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.964728,0.031271,0.867635,1.000000
synonyms,0.860048,0.062026,0.659849,0.984292


In [26]:
results_df['jaccard_lemmas']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.913287,0.056302,0.789474,1.000000
synonyms,0.613378,0.126446,0.350000,0.904762


In [27]:
results_df['common_words_ratio']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.953559,0.031037,0.882353,1.00
synonyms,0.752889,0.097835,0.518519,0.95


In [28]:
results_df['levenshtein_distance']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.045328,0.029588,0.006803,0.131944
synonyms,0.187443,0.075453,0.025126,0.387640


In [29]:
results_df['levenshtein_similarity']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.954672,0.029588,0.868056,0.993197
synonyms,0.812557,0.075453,0.612360,0.974874


In [30]:
results_df['rouge_l']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.954428,0.032350,0.857143,1.000000
synonyms,0.758489,0.097445,0.517241,0.952381


In [31]:
results_df['bleu']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.851597,0.090131,0.572951,1.000000
synonyms,0.537449,0.163683,0.132245,0.878742


In [32]:
results_df['chrf']

,mean,std,min,max
augmentation_type,,,,
antonyms,0.944989,0.039916,0.846647,1.000000
synonyms,0.744876,0.095708,0.500138,0.951377
